# Multiclass XGBoost 8-class training with strong simulated patterns

本 notebook 基于原始 8 分类训练脚本修改，**其他训练流程保持不变**，只在训练数据构建阶段新增一条筛选规则：

```python
proportion_pattern >= 0.75
```

也就是说，最终用于训练和 5 折 GroupKFold 验证的数据只来自模拟数据中模式强度较高的 cell-gene 样本。

保留原流程：
- 从 `simulated_data_dict.pkl` 的 `data_df` 聚合 `groundtruth -> y_true`
- 和 `simulated_features.parquet` 合并
- 5 折 `GroupKFold`，按 `cell` 分组
- 单标签 8 分类 `XGBoost multiclass`
- 输出 OOF、fold metrics、summary、per-class metrics、confusion matrix
- 最后用筛选后的数据重训 final model，并保存 `joblib` 和 final parquet

本 notebook **不做**：
- OVR
- per-class threshold
- pattern_multi
- uncertainty


In [1]:
# %%
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import json
import joblib

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
)

from xgboost import XGBClassifier

ROOT = Path("/home/lixiangyu/xwq/Subcellular_Feature")

PKL_PATH = ROOT / "data" / "simulated_data_dict.pkl"
FEAT_PATH = ROOT / "data" / "simulated_features.parquet"

OOF_OUT_PARQ = ROOT / "data" / "multiclass_xgb_prop075_group5cv_oof.parquet"
FOLD_METRICS_CSV = ROOT / "data" / "multiclass_xgb_prop075_group5cv_fold_metrics.csv"
SUMMARY_CSV = ROOT / "data" / "multiclass_xgb_prop075_group5cv_summary.csv"
PER_CLASS_CSV = ROOT / "data" / "multiclass_xgb_prop075_group5cv_per_class_metrics.csv"
OVERALL_JSON = ROOT / "data" / "multiclass_xgb_prop075_group5cv_overall_metrics.json"
CM_DIR = ROOT / "data" / "multiclass_xgb_prop075_group5cv_confusion_matrices"

FINAL_MODEL_PATH = ROOT / "models" / "multiclass_xgb_8class_prop075_final_from_cv.joblib"
FINAL_OUT_PARQ = ROOT / "data" / "simulated_features_with_multiclass_xgb_prop075_final.parquet"

# Only simulated samples with proportion_pattern >= this value are used for training.
STRONG_PATTERN_MIN = 0.75

CM_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

In [2]:
# %%
obj = pickle.load(open(PKL_PATH, "rb"))
spot = obj["data_df"].copy()
feat = pd.read_parquet(FEAT_PATH).copy()

if "proportion_pattern" not in spot.columns:
    raise ValueError(
        "data_df 中没有 proportion_pattern 列，无法执行 proportion_pattern >= 0.75 的强模式筛选。"
    )

for c in ["cell", "gene"]:
    spot[c] = spot[c].astype(str).str.strip()
    feat[c] = feat[c].astype(str).str.strip()

spot["groundtruth"] = spot["groundtruth"].astype(str).str.strip()
spot["proportion_pattern"] = pd.to_numeric(spot["proportion_pattern"], errors="coerce")

# ---------------------------------------------------------------------
# New filtering step:
# Build cell-gene level metadata first, then keep only strong simulated
# samples with proportion_pattern >= STRONG_PATTERN_MIN.
#
# One training sample = one (cell, gene).
# proportion_pattern is averaged within each (cell, gene).
# groundtruth uses the dominant label if multiple labels exist.
# ---------------------------------------------------------------------
cellgene_meta = (
    spot.groupby(["cell", "gene"])
    .agg(
        y_true=("groundtruth", lambda s: s.value_counts().idxmax()),
        proportion_pattern=("proportion_pattern", "mean"),
        n_transcripts=("gene", "size"),
    )
    .reset_index()
)

print("cellgene_meta before strength filtering:", cellgene_meta.shape)
print("\nproportion_pattern distribution before filtering:")
print(cellgene_meta["proportion_pattern"].describe())
print("\nproportion_pattern value counts before filtering:")
print(cellgene_meta["proportion_pattern"].value_counts().sort_index())
print("\nclass distribution before filtering:")
print(cellgene_meta["y_true"].value_counts())

cellgene_meta = cellgene_meta[
    cellgene_meta["proportion_pattern"] >= STRONG_PATTERN_MIN
].copy()

print("\ncellgene_meta after strength filtering:", cellgene_meta.shape)
print(f"kept proportion_pattern >= {STRONG_PATTERN_MIN}")
print("\nproportion_pattern distribution after filtering:")
print(cellgene_meta["proportion_pattern"].describe())
print("\nproportion_pattern value counts after filtering:")
print(cellgene_meta["proportion_pattern"].value_counts().sort_index())
print("\nclass distribution after filtering:")
print(cellgene_meta["y_true"].value_counts())

# Keep original downstream structure: y_df only provides cell/gene/y_true.
y_df = cellgene_meta[["cell", "gene", "y_true"]].copy()

data = feat.merge(y_df, on=["cell", "gene"], how="inner")

print("\nspot :", spot.shape)
print("feat :", feat.shape)
print("data :", data.shape)
print()
print(data["y_true"].value_counts())
data.head()


cellgene_meta before strength filtering: (20000, 5)

proportion_pattern distribution before filtering:
count    20000.000000
mean         0.775000
std          0.143618
min          0.550000
25%          0.650000
50%          0.775000
75%          0.900000
max          1.000000
Name: proportion_pattern, dtype: float64

proportion_pattern value counts before filtering:
proportion_pattern
0.55    2000
0.60    2000
0.65    1985
0.65      15
0.70      21
0.70    1979
0.75    2000
0.80      16
0.80    1943
0.80      41
0.85      48
0.85    1936
0.85      16
0.90      41
0.90    1908
0.90      51
0.95      73
0.95    1890
0.95      37
1.00    2000
Name: count, dtype: int64

class distribution before filtering:
y_true
Random          2500
Foci            2500
Nuclear         2500
Cytoplasmic     2500
Nuclear edge    2500
Cell edge       2500
Protrusion      2500
Radial          2500
Name: count, dtype: int64

cellgene_meta after strength filtering: (12000, 5)
kept proportion_pattern >= 0.75



,cell,gene,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,bento_nucleus_dispersion_norm,bento_l_max,bento_l_max_gradient,bento_l_min_gradient,bento_l_monotony,bento_l_half_radius,sprawl_peripheral,sprawl_central,sprawl_punctate,sprawl_radial,y_true
0,cell_102,gene_0_4_0,0.606768,0.803927,0.832587,0.505908,0.345752,0.717048,0.563189,0.631452,6.088981,2.621934,-1.003448,-0.841019,0.102425,-0.313444,0.377245,0.49,-0.70,Random
1,cell_102,gene_0_4_1,0.626388,0.791865,0.822830,0.544798,0.333810,0.719395,0.594613,0.553160,3.143470,0.384844,-0.855165,-0.839041,-0.858051,-0.188585,0.198993,0.86,-0.47,Random
2,cell_102,gene_0_4_2,0.632985,0.833599,0.709496,0.548204,0.366426,0.843104,0.682718,0.786127,2.360197,0.453603,-0.805518,-0.882414,-1.697862,-0.282633,0.292628,0.40,0.83,Random
3,cell_102,gene_0_4_3,0.651571,0.825118,0.757306,0.558804,0.357483,0.784780,0.661902,0.658709,3.885493,0.218315,-0.814964,-0.911195,-1.884277,-0.215563,0.289847,-0.05,0.18,Random
4,cell_102,gene_0_4_4,0.650200,0.810594,0.738138,0.576106,0.349152,0.809373,0.719623,0.696350,1.704610,0.257475,-0.759488,-0.916358,-3.636081,-0.295484,0.236041,0.17,-0.83,Random


In [3]:
# %%
CLASSES = [
    "Foci",
    "Nuclear",
    "Cytoplasmic",
    "Nuclear edge",
    "Cell edge",
    "Protrusion",
    "Radial",
    "Random",
]

FEATURE_COLS = [
    c for c in data.columns
    if c not in ("cell", "gene", "y_true")
]

CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}

print("n_features:", len(FEATURE_COLS))
print(FEATURE_COLS)
print("class_to_id:", CLASS_TO_ID)

n_features: 17
['bento_cell_inner_proximity', 'bento_nucleus_inner_proximity', 'bento_nucleus_outer_proximity', 'bento_cell_inner_asymmetry', 'bento_nucleus_inner_asymmetry', 'bento_nucleus_outer_asymmetry', 'bento_point_dispersion_norm', 'bento_nucleus_dispersion_norm', 'bento_l_max', 'bento_l_max_gradient', 'bento_l_min_gradient', 'bento_l_monotony', 'bento_l_half_radius', 'sprawl_peripheral', 'sprawl_central', 'sprawl_punctate', 'sprawl_radial']
class_to_id: {'Foci': 0, 'Nuclear': 1, 'Cytoplasmic': 2, 'Nuclear edge': 3, 'Cell edge': 4, 'Protrusion': 5, 'Radial': 6, 'Random': 7}


In [4]:
# %%
def make_multiclass_xgb(random_state=42):
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(CLASSES),
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        eval_metric="mlogloss",
        random_state=random_state,
        n_jobs=-1,
        tree_method="hist",
    )
    return model

In [5]:
# %%
X = data[FEATURE_COLS].copy()
y = data["y_true"].astype(str).copy()
y_id = y.map(CLASS_TO_ID).astype(int)
groups = data["cell"].astype(str).values

gkf = GroupKFold(n_splits=5)

fold_metrics = []
oof_rows = []
fold_confusion_matrices = {}

In [6]:
# %%
for fold_id, (train_idx, test_idx) in enumerate(gkf.split(X, y_id, groups=groups), start=1):
    print(f"\n========== Fold {fold_id} ==========")

    X_train = X.iloc[train_idx].reset_index(drop=True)
    y_train = y_id.iloc[train_idx].reset_index(drop=True)
    cell_train = data["cell"].iloc[train_idx].reset_index(drop=True)

    X_test = X.iloc[test_idx].reset_index(drop=True)
    y_test = y_id.iloc[test_idx].reset_index(drop=True)
    cell_test = data["cell"].iloc[test_idx].reset_index(drop=True)

    print("train:", X_train.shape, "test:", X_test.shape)
    print("train cells:", cell_train.nunique(), "test cells:", cell_test.nunique())

    model = make_multiclass_xgb(random_state=42 + fold_id)
    model.fit(X_train, y_train)

    prob_test = model.predict_proba(X_test)
    pred_test_id = model.predict(X_test)
    pred_test = pd.Series(pred_test_id).map(ID_TO_CLASS).values
    y_test_label = pd.Series(y_test).map(ID_TO_CLASS).values

    prob_df = pd.DataFrame(prob_test, columns=[f"p_{ID_TO_CLASS[i]}" for i in range(len(CLASSES))])
    prob_df = prob_df.reindex(columns=[f"p_{c}" for c in CLASSES])

    acc = accuracy_score(y_test_label, pred_test)
    bal_acc = balanced_accuracy_score(y_test_label, pred_test)

    report = classification_report(
        y_test_label,
        pred_test,
        labels=CLASSES,
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(y_test_label, pred_test, labels=CLASSES)
    cm_df = pd.DataFrame(
        cm,
        index=[f"true:{c}" for c in CLASSES],
        columns=[f"pred:{c}" for c in CLASSES]
    )
    fold_confusion_matrices[fold_id] = cm_df
    cm_df.to_csv(CM_DIR / f"fold_{fold_id}_confusion_matrix.csv", encoding="utf-8-sig")

    row = {
        "fold": fold_id,
        "n_test": len(y_test),
        "n_test_cells": cell_test.nunique(),
        "accuracy": float(acc),
        "balanced_accuracy": float(bal_acc),
        "macro_precision": report["macro avg"]["precision"],
        "macro_recall": report["macro avg"]["recall"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_precision": report["weighted avg"]["precision"],
        "weighted_recall": report["weighted avg"]["recall"],
        "weighted_f1": report["weighted avg"]["f1-score"],
    }

    for c in CLASSES:
        row[f"precision_{c}"] = report.get(c, {}).get("precision", np.nan)
        row[f"recall_{c}"] = report.get(c, {}).get("recall", np.nan)
        row[f"f1_{c}"] = report.get(c, {}).get("f1-score", np.nan)

    fold_metrics.append(row)

    fold_out = data.iloc[test_idx].copy().reset_index(drop=True)
    fold_out["fold"] = fold_id

    for c in CLASSES:
        fold_out[f"p_{c}"] = prob_df[f"p_{c}"].values

    fold_out["pattern_top1"] = pred_test
    oof_rows.append(fold_out)

    print("fold accuracy:", acc)
    print("fold balanced_accuracy:", bal_acc)


========== Fold 1 ==========
train: (9600, 17) test: (2400, 17)
train cells: 40 test cells: 10
fold accuracy: 0.9483333333333334
fold balanced_accuracy: 0.9483333333333334

========== Fold 2 ==========
train: (9600, 17) test: (2400, 17)
train cells: 40 test cells: 10
fold accuracy: 0.9566666666666667
fold balanced_accuracy: 0.9566666666666667

========== Fold 3 ==========
train: (9600, 17) test: (2400, 17)
train cells: 40 test cells: 10
fold accuracy: 0.96625
fold balanced_accuracy: 0.9662499999999999

========== Fold 4 ==========
train: (9600, 17) test: (2400, 17)
train cells: 40 test cells: 10
fold accuracy: 0.965
fold balanced_accuracy: 0.965

========== Fold 5 ==========
train: (9600, 17) test: (2400, 17)
train cells: 40 test cells: 10
fold accuracy: 0.9470833333333334
fold balanced_accuracy: 0.9470833333333334


In [7]:
# %%
fold_metrics_df = pd.DataFrame(fold_metrics)
oof_df = pd.concat(oof_rows, axis=0, ignore_index=True)

oof_df = oof_df.sort_values(["fold", "cell", "gene"]).reset_index(drop=True)

fold_metrics_df.to_csv(FOLD_METRICS_CSV, index=False, encoding="utf-8-sig")
oof_df.to_parquet(OOF_OUT_PARQ, index=False)

print("saved:", FOLD_METRICS_CSV)
print("saved:", OOF_OUT_PARQ)

print("oof_df shape:", oof_df.shape)
oof_df.head()

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_fold_metrics.csv
saved: /home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_oof.parquet
oof_df shape: (12000, 30)


,cell,gene,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,bento_nucleus_dispersion_norm,...,fold,p_Foci,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random,pattern_top1
0,cell_135,gene_0_4_0,0.702716,0.795730,0.761774,0.613912,0.284685,0.808014,0.718738,0.694958,...,1,0.002589,0.000184,0.163289,0.000666,0.000193,0.000205,0.000251,0.832622,Random
1,cell_135,gene_0_4_1,0.690424,0.834431,0.752051,0.612681,0.348335,0.805851,0.678040,0.664558,...,1,0.000152,0.000046,0.003392,0.000436,0.000070,0.000052,0.000051,0.995800,Random
2,cell_135,gene_0_4_2,0.708866,0.847627,0.755122,0.667060,0.342497,0.808470,0.873987,0.834232,...,1,0.000604,0.000215,0.678061,0.000549,0.001197,0.000242,0.000262,0.318869,Cytoplasmic
3,cell_135,gene_0_4_3,0.676781,0.836623,0.795318,0.519532,0.340545,0.742967,0.564167,0.552959,...,1,0.000156,0.000076,0.002073,0.018965,0.000081,0.000085,0.000098,0.978467,Random
4,cell_135,gene_0_4_4,0.698068,0.831448,0.730982,0.597128,0.344317,0.818814,0.740143,0.739584,...,1,0.000082,0.000045,0.016336,0.000224,0.000042,0.000050,0.000054,0.983167,Random


In [8]:
# %%
oof_acc = (oof_df["y_true"] == oof_df["pattern_top1"]).mean()
oof_bal_acc = balanced_accuracy_score(oof_df["y_true"], oof_df["pattern_top1"])

oof_report = classification_report(
    oof_df["y_true"],
    oof_df["pattern_top1"],
    labels=CLASSES,
    output_dict=True,
    zero_division=0
)

overall_metrics = {
    "oof_accuracy": float(oof_acc),
    "oof_balanced_accuracy": float(oof_bal_acc),
    "macro_precision": float(oof_report["macro avg"]["precision"]),
    "macro_recall": float(oof_report["macro avg"]["recall"]),
    "macro_f1": float(oof_report["macro avg"]["f1-score"]),
    "weighted_precision": float(oof_report["weighted avg"]["precision"]),
    "weighted_recall": float(oof_report["weighted avg"]["recall"]),
    "weighted_f1": float(oof_report["weighted avg"]["f1-score"]),
}

with open(OVERALL_JSON, "w", encoding="utf-8") as f:
    json.dump(overall_metrics, f, ensure_ascii=False, indent=2)

print("saved:", OVERALL_JSON)
print("OOF accuracy:", oof_acc)
print("OOF balanced accuracy:", oof_bal_acc)
print()
print(classification_report(
    oof_df["y_true"],
    oof_df["pattern_top1"],
    labels=CLASSES,
    digits=4,
    zero_division=0
))

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_overall_metrics.json
OOF accuracy: 0.9566666666666667
OOF balanced accuracy: 0.9566666666666667

              precision    recall  f1-score   support

        Foci     0.9598    0.9547    0.9572      1500
     Nuclear     0.9953    0.9927    0.9940      1500
 Cytoplasmic     0.8807    0.8960    0.8883      1500
Nuclear edge     0.9624    0.9560    0.9592      1500
   Cell edge     0.9728    0.9760    0.9744      1500
  Protrusion     0.9973    0.9793    0.9882      1500
      Radial     0.9886    0.9853    0.9870      1500
      Random     0.8995    0.9133    0.9064      1500

    accuracy                         0.9567     12000
   macro avg     0.9571    0.9567    0.9568     12000
weighted avg     0.9571    0.9567    0.9568     12000



In [9]:
# %%
metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_precision",
    "weighted_recall",
    "weighted_f1",
] + [f"f1_{c}" for c in CLASSES]

summary_df = pd.DataFrame({
    "metric": metric_cols,
    "mean": [fold_metrics_df[c].mean() for c in metric_cols],
    "std": [fold_metrics_df[c].std(ddof=1) for c in metric_cols],
})

summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("saved:", SUMMARY_CSV)
summary_df

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_summary.csv


,metric,mean,std
0,accuracy,0.956667,0.008980
1,balanced_accuracy,0.956667,0.008980
2,macro_precision,0.957579,0.007997
3,macro_recall,0.956667,0.008980
4,macro_f1,0.956814,0.008772
5,weighted_precision,0.957579,0.007997
6,weighted_recall,0.956667,0.008980
7,weighted_f1,0.956814,0.008772
8,f1_Foci,0.957412,0.014649
9,f1_Nuclear,0.993986,0.003053


In [10]:
# %%
per_class_rows = []
for c in CLASSES:
    per_class_rows.append({
        "class": c,
        "precision": oof_report[c]["precision"],
        "recall": oof_report[c]["recall"],
        "f1": oof_report[c]["f1-score"],
        "support": oof_report[c]["support"],
    })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(PER_CLASS_CSV, index=False, encoding="utf-8-sig")
print("saved:", PER_CLASS_CSV)
per_class_df

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_per_class_metrics.csv


,class,precision,recall,f1,support
0,Foci,0.959786,0.954667,0.957219,1500.0
1,Nuclear,0.995321,0.992667,0.993992,1500.0
2,Cytoplasmic,0.880734,0.896000,0.888301,1500.0
3,Nuclear edge,0.962416,0.956000,0.959197,1500.0
4,Cell edge,0.972757,0.976000,0.974376,1500.0
5,Protrusion,0.997284,0.979333,0.988227,1500.0
6,Radial,0.988629,0.985333,0.986978,1500.0
7,Random,0.899540,0.913333,0.906384,1500.0


In [11]:
# %%
cm_oof = confusion_matrix(
    oof_df["y_true"],
    oof_df["pattern_top1"],
    labels=CLASSES
)

cm_oof_df = pd.DataFrame(
    cm_oof,
    index=[f"true:{c}" for c in CLASSES],
    columns=[f"pred:{c}" for c in CLASSES]
)

OOF_CM_CSV = CM_DIR / "oof_confusion_matrix.csv"
cm_oof_df.to_csv(OOF_CM_CSV, encoding="utf-8-sig")
print("saved:", OOF_CM_CSV)
cm_oof_df

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_confusion_matrices/oof_confusion_matrix.csv


,pred:Foci,pred:Nuclear,pred:Cytoplasmic,pred:Nuclear edge,pred:Cell edge,pred:Protrusion,pred:Radial,pred:Random
true:Foci,1432,0,49,5,3,3,1,7
true:Nuclear,0,1489,0,0,0,0,2,9
true:Cytoplasmic,30,1,1344,13,22,0,0,90
true:Nuclear edge,4,1,7,1434,0,0,7,47
true:Cell edge,1,0,33,0,1464,0,2,0
true:Protrusion,23,0,0,0,3,1469,5,0
true:Radial,2,1,0,5,13,1,1478,0
true:Random,0,4,93,33,0,0,0,1370


In [12]:
# %%
X_all = data[FEATURE_COLS].reset_index(drop=True).copy()
y_all = data["y_true"].astype(str).reset_index(drop=True).copy()
y_all_id = y_all.map(CLASS_TO_ID).astype(int)

final_model = make_multiclass_xgb(random_state=42)
final_model.fit(X_all, y_all_id)

print("trained final multiclass XGBoost model")

trained final multiclass XGBoost model


In [13]:
# %%
prob_all = final_model.predict_proba(X_all)
pred_all_id = final_model.predict(X_all)
pred_all = pd.Series(pred_all_id).map(ID_TO_CLASS).values

prob_df = pd.DataFrame(prob_all, columns=[f"p_{ID_TO_CLASS[i]}" for i in range(len(CLASSES))])
prob_df = prob_df.reindex(columns=[f"p_{c}" for c in CLASSES])

out = data.copy().reset_index(drop=True)

for c in CLASSES:
    out[f"p_{c}"] = prob_df[f"p_{c}"].values

out["pattern_top1"] = pred_all

base_cols = ["cell", "gene", "y_true"]
prob_cols = [f"p_{c}" for c in CLASSES]
final_cols = base_cols + FEATURE_COLS + prob_cols + ["pattern_top1"]
out = out[final_cols]

FINAL_OUT_PARQ.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(FINAL_OUT_PARQ, index=False)

print("saved:", FINAL_OUT_PARQ)
print(out.shape)
out.head()

saved: /home/lixiangyu/xwq/Subcellular_Feature/data/simulated_features_with_multiclass_xgb_prop075_final.parquet
(12000, 29)


,cell,gene,y_true,bento_cell_inner_proximity,bento_nucleus_inner_proximity,bento_nucleus_outer_proximity,bento_cell_inner_asymmetry,bento_nucleus_inner_asymmetry,bento_nucleus_outer_asymmetry,bento_point_dispersion_norm,...,sprawl_radial,p_Foci,p_Nuclear,p_Cytoplasmic,p_Nuclear edge,p_Cell edge,p_Protrusion,p_Radial,p_Random,pattern_top1
0,cell_102,gene_0_4_0,Random,0.606768,0.803927,0.832587,0.505908,0.345752,0.717048,0.563189,...,-0.70,0.014871,0.000130,0.029090,0.020333,0.000195,0.000111,0.000152,0.935118,Random
1,cell_102,gene_0_4_1,Random,0.626388,0.791865,0.822830,0.544798,0.333810,0.719395,0.594613,...,-0.47,0.000045,0.000019,0.000444,0.000142,0.000023,0.000020,0.000022,0.999286,Random
2,cell_102,gene_0_4_2,Random,0.632985,0.833599,0.709496,0.548204,0.366426,0.843104,0.682718,...,0.83,0.000230,0.000126,0.020980,0.001014,0.000135,0.000129,0.000167,0.977220,Random
3,cell_102,gene_0_4_3,Random,0.651571,0.825118,0.757306,0.558804,0.357483,0.784780,0.661902,...,0.18,0.000022,0.000016,0.000740,0.000046,0.000018,0.000017,0.000018,0.999123,Random
4,cell_102,gene_0_4_4,Random,0.650200,0.810594,0.738138,0.576106,0.349152,0.809373,0.719623,...,-0.83,0.000028,0.000021,0.001278,0.000066,0.000024,0.000022,0.000024,0.998535,Random


In [14]:
# %%
save_obj = {
    "model": final_model,
    "feature_cols": FEATURE_COLS,
    "classes": CLASSES,
    "class_to_id": CLASS_TO_ID,
    "id_to_class": ID_TO_CLASS,
    "meta": {
        "trained_on": "simulated data with proportion_pattern >= 0.75 after GroupKFold 5-fold CV evaluation",
        "task_type": "single-label multiclass classification",
        "model_type": "xgboost_multiclass",
        "n_samples": int(len(data)),
        "n_cells": int(data["cell"].nunique()),
        "n_genes": int(data["gene"].nunique()),
        "model_version": "multiclass_xgb_prop075_v1",
        "strong_pattern_filter": {"proportion_pattern_min": float(STRONG_PATTERN_MIN)},
        "xgb_params": {
            "objective": "multi:softprob",
            "num_class": len(CLASSES),
            "n_estimators": 300,
            "learning_rate": 0.05,
            "max_depth": 6,
            "min_child_weight": 1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.0,
            "reg_lambda": 1.0,
            "eval_metric": "mlogloss",
            "random_state": 42,
            "n_jobs": -1,
            "tree_method": "hist",
        },
        "oof_metrics": overall_metrics,
    }
}

joblib.dump(save_obj, FINAL_MODEL_PATH)
print("saved final model:", FINAL_MODEL_PATH)

saved final model: /home/lixiangyu/xwq/Subcellular_Feature/models/multiclass_xgb_8class_prop075_final_from_cv.joblib


In [15]:
import pandas as pd

df = pd.read_parquet("/home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_oof.parquet")

same_count = (df["y_true"] == df["pattern_top1"]).sum()
total_count = len(df)

print("y_true 与 pattern_top1 相同的数目:", same_count)
print("总样本数:", total_count)
print("占比:", same_count / total_count)

y_true 与 pattern_top1 相同的数目: 11480
总样本数: 12000
占比: 0.9566666666666667


In [16]:
OOF_PATH = "/home/lixiangyu/xwq/Subcellular_Feature/data/multiclass_xgb_prop075_group5cv_oof.parquet"

df = pd.read_parquet(OOF_PATH)

# 1) 相同数目
same_count = (df["y_true"] == df["pattern_top1"]).sum()
print("y_true 与 pattern_top1 相同的数目:", same_count)

# 2) 不同的条目
diff_df = df[df["y_true"] != df["pattern_top1"]].copy()

print("y_true 与 pattern_top1 不同的条数:", len(diff_df))
diff_view = diff_df[["cell", "gene", "y_true", "pattern_top1"]].copy()
diff_view.head(50)

y_true 与 pattern_top1 相同的数目: 11480
y_true 与 pattern_top1 不同的条数: 520


,cell,gene,y_true,pattern_top1
2,cell_135,gene_0_4_2,Random,Cytoplasmic
12,cell_135,gene_0_6_2,Random,Cytoplasmic
15,cell_135,gene_0_7_0,Random,Cytoplasmic
51,cell_135,gene_1_8_1,Foci,Cytoplasmic
58,cell_135,gene_1_9_3,Foci,Cytoplasmic
90,cell_135,gene_3_4_0,Cytoplasmic,Random
100,cell_135,gene_3_6_0,Cytoplasmic,Foci
116,cell_135,gene_3_9_1,Cytoplasmic,Cell edge
135,cell_135,gene_4_7_0,Nuclear edge,Random
140,cell_135,gene_4_8_0,Nuclear edge,Random


In [17]:
from tools.models.pattern_classifier import PatternClassifier

clf = PatternClassifier("../models/multiclass_xgb_8class_prop075_final_from_cv.joblib")
print(clf.classes)
print(clf.feature_cols[:5])
print(type(clf.model))

['Foci', 'Nuclear', 'Cytoplasmic', 'Nuclear edge', 'Cell edge', 'Protrusion', 'Radial', 'Random']
['bento_cell_inner_proximity', 'bento_nucleus_inner_proximity', 'bento_nucleus_outer_proximity', 'bento_cell_inner_asymmetry', 'bento_nucleus_inner_asymmetry']
<class 'xgboost.sklearn.XGBClassifier'>
